In [0]:
# Silver Layer: Date and Time Transformations
# Author: Virendra Dilip Tambavekar
# HRM ID: 6217
# Domain: Cross-Domain Reference
# Source: bronze.date, bronze.time
# Target: silver.date, silver.time
# Description: Multi-format date/time parsing, quarantine invalid records,
#              derive calendar and time dimension fields.

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, lower, when, lit, coalesce,
    to_date, date_format, year, month, dayofweek,
    quarter, weekofyear, concat, lpad,
    hour, minute, second, current_timestamp,
    row_number, regexp_replace, length
)
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, IntegerType, BooleanType, TimestampType

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG = "charles_schwab_retailbrokerage_dev_team_lemma"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"
DOMAIN = "cross"
BATCH = "Batch1"

# Set catalog context for operations utility functions
spark.sql(f"USE CATALOG {CATALOG}")

# Retrieve run_id from bronze table (reuse existing run_id, do not generate new)
run_id_row = spark.sql(f"""
    SELECT DISTINCT _run_id 
    FROM {CATALOG}.{BRONZE_SCHEMA}.date 
    ORDER BY _run_id DESC 
    LIMIT 1
""").collect()

RUN_ID = run_id_row[0]["_run_id"]
print(f"Reusing run_id from bronze: {RUN_ID}")

# Start pipeline run
start_pipeline_run(spark, RUN_ID, BATCH)
log_pipeline_message(spark, RUN_ID, "INFO", "silver_date_and_time", "Silver layer processing started for Date and Time tables")

In [0]:
# ---------------------------------------------------------------------------
# SILVER DATE: Read from Bronze
# ---------------------------------------------------------------------------
bronze_date_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.date")
bronze_date_count = bronze_date_df.count()
print(f"Bronze date record count: {bronze_date_count}")

# ---------------------------------------------------------------------------
# Step 1: Trim whitespace from source_date_string and apply multi-format parsing
# Formats by source_system_code:
#   MF-LEGACY : DD-MON-YY   (e.g., '01-JAN-50')
#   CRM-01    : MM/dd/yyyy  (e.g., '01/02/1950')
#   ERP-SAP   : yyyy.MM.dd  (e.g., '1950.01.03')
#   TRD-DESK  : English     (e.g., 'January 10, 1950', 'Jan 11, 1950')
# ---------------------------------------------------------------------------
from pyspark.sql.functions import expr, add_months

date_parsed_df = bronze_date_df.withColumn(
    "_trimmed_date", trim(col("source_date_string"))
).withColumn(
    "_raw_date",
    coalesce(
        # CRM-01: MM/dd/yyyy
        when(
            col("source_system_code") == "CRM-01",
            expr("try_to_date(_trimmed_date, 'MM/dd/yyyy')")
        ),
        # ERP-SAP: yyyy.MM.dd
        when(
            col("source_system_code") == "ERP-SAP",
            expr("try_to_date(_trimmed_date, 'yyyy.MM.dd')")
        ),
        # MF-LEGACY: dd-MMM-yy (2-digit year)
        when(
            col("source_system_code") == "MF-LEGACY",
            expr("try_to_date(upper(_trimmed_date), 'dd-MMM-yy')")
        ),
        # TRD-DESK: Multiple English formats - try full month name then abbreviated
        when(
            col("source_system_code") == "TRD-DESK",
            coalesce(
                expr("try_to_date(_trimmed_date, 'MMMM d, yyyy')"),
                expr("try_to_date(_trimmed_date, 'MMMM dd, yyyy')"),
                expr("try_to_date(_trimmed_date, 'MMM d, yyyy')"),
                expr("try_to_date(_trimmed_date, 'MMM dd, yyyy')")
            )
        )
    )
).withColumn(
    # Year correction for MF-LEGACY 2-digit years:
    # Spark interprets yy=50 as 2050 (wrong). Correct: if year > 2020, subtract 100.
    "DateValue",
    when(
        year(col("_raw_date")) > 2020,
        add_months(col("_raw_date"), -1200)  # subtract 100 years (1200 months)
    ).otherwise(col("_raw_date"))
).drop("_trimmed_date", "_raw_date")

# ---------------------------------------------------------------------------
# Step 2: Deduplicate by DateValue (keep latest by _ingest_ts)
# ---------------------------------------------------------------------------
date_window = Window.partitionBy("DateValue").orderBy(col("_ingest_ts").desc())

date_dedup_df = date_parsed_df.filter(
    col("DateValue").isNotNull()
).withColumn(
    "_row_num", row_number().over(date_window)
).filter(
    col("_row_num") == 1
).drop("_row_num")

# ---------------------------------------------------------------------------
# Step 3: Quarantine - parse failures and duplicates
# ---------------------------------------------------------------------------
date_quarantine_df = date_parsed_df.filter(
    col("DateValue").isNull()
).select(
    col("record_id"),
    col("source_date_string"),
    col("source_system_code"),
    lit("PARSE_FAILED").alias("_reject_reason"),
    current_timestamp().alias("_quarantined_at"),
    lit(RUN_ID).alias("_run_id"),
    col("_batch_id").alias("_batch")
)

# Also quarantine duplicates (rows that lost in dedup)
date_duplicates_df = date_parsed_df.filter(
    col("DateValue").isNotNull()
).withColumn(
    "_row_num", row_number().over(date_window)
).filter(
    col("_row_num") > 1
).drop("_row_num").select(
    col("record_id"),
    col("source_date_string"),
    col("source_system_code"),
    lit("DUPLICATE").alias("_reject_reason"),
    current_timestamp().alias("_quarantined_at"),
    lit(RUN_ID).alias("_run_id"),
    col("_batch_id").alias("_batch")
)

date_quarantine_all = date_quarantine_df.unionByName(date_duplicates_df)
quarantine_date_count = date_quarantine_all.count()
print(f"Date records quarantined: {quarantine_date_count}")

# ---------------------------------------------------------------------------
# Step 4: Derive calendar dimension fields
# ---------------------------------------------------------------------------
# Fiscal year starts July 1 (July 2020 = FY2021)
silver_date_df = date_dedup_df.select(
    col("DateValue"),
    date_format(col("DateValue"), "MMMM d, yyyy").alias("DateDesc"),
    year(col("DateValue")).cast(IntegerType()).alias("CalendarYearID"),
    concat(lit("CY"), year(col("DateValue")).cast("string")).alias("CalendarYearDesc"),
    (year(col("DateValue")) * 10 + quarter(col("DateValue"))).cast(IntegerType()).alias("CalendarQtrID"),
    concat(
        year(col("DateValue")).cast("string"), lit(" Q"), quarter(col("DateValue")).cast("string")
    ).alias("CalendarQtrDesc"),
    (year(col("DateValue")) * 100 + month(col("DateValue"))).cast(IntegerType()).alias("CalendarMonthID"),
    date_format(col("DateValue"), "MMMM yyyy").alias("CalendarMonthDesc"),
    (year(col("DateValue")) * 100 + weekofyear(col("DateValue"))).cast(IntegerType()).alias("CalendarWeekID"),
    concat(
        year(col("DateValue")).cast("string"), lit(" W"),
        lpad(weekofyear(col("DateValue")).cast("string"), 2, "0")
    ).alias("CalendarWeekDesc"),
    # DayOfWeekNum: Spark dayofweek returns 1=Sun,2=Mon,...,7=Sat. ISO: 1=Mon,...,7=Sun
    when(dayofweek(col("DateValue")) == 1, 7)
    .otherwise(dayofweek(col("DateValue")) - 1).cast(IntegerType()).alias("DayOfWeekNum"),
    date_format(col("DateValue"), "EEEE").alias("DayOfWeekDesc"),
    # Fiscal Year: starts July 1 (month >= 7 means next fiscal year)
    when(
        month(col("DateValue")) >= 7,
        year(col("DateValue")) + 1
    ).otherwise(year(col("DateValue"))).cast(IntegerType()).alias("FiscalYearID"),
    when(
        month(col("DateValue")) >= 7,
        concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))
    ).otherwise(
        concat(lit("FY"), year(col("DateValue")).cast("string"))
    ).alias("FiscalYearDesc"),
    # Fiscal Quarter: Jul-Sep=FQ1, Oct-Dec=FQ2, Jan-Mar=FQ3, Apr-Jun=FQ4
    when(month(col("DateValue")).between(7, 9), 1)
    .when(month(col("DateValue")).between(10, 12), 2)
    .when(month(col("DateValue")).between(1, 3), 3)
    .otherwise(4).cast(IntegerType()).alias("FiscalQtrID"),
    when(month(col("DateValue")).between(7, 9),
         concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q1")))
    .when(month(col("DateValue")).between(10, 12),
         concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q2")))
    .when(month(col("DateValue")).between(1, 3),
         concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q3")))
    .otherwise(
         concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q4")))
    .alias("FiscalQtrDesc"),
    # HolidayFlag: placeholder (set to False; can be enriched later with holiday calendar)
    lit(False).cast(BooleanType()).alias("HolidayFlag"),
    # Audit columns
    col("_batch_id").alias("_batch"),
    current_timestamp().alias("_load_ts"),
    lit(RUN_ID).alias("_run_id")
)

silver_date_count = silver_date_df.count()
print(f"Silver date record count: {silver_date_count}")

In [0]:
# ---------------------------------------------------------------------------
# SILVER TIME: Bronze to Silver Transformation
# ---------------------------------------------------------------------------
# Approach:
#   1. Parse and normalize multi-format time strings
#   2. Quarantine UNKNOWN precision, NULL_TIME, and DUPLICATE_BATCH records
#   3. EXPLODE MINUTE precision records into 60 second-level rows each
#   4. Union SECOND + expanded MINUTE records
#   5. Dedup by standard_time to get exactly 86,400 unique seconds
# ---------------------------------------------------------------------------
from pyspark.sql.functions import (
    expr, substring, to_timestamp, regexp_extract, explode,
    sequence, concat_ws, try_to_timestamp
)

bronze_time_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time")
bronze_time_count = bronze_time_df.count()
print(f"Bronze time record count: {bronze_time_count}")

# Dedup by record_id as per problem statement
bronze_time_df = bronze_time_df.dropDuplicates(["record_id"])

quarantine_parts = []

base_cols = ["record_id", "source_time_string", "source_system_code",
             "time_precision", "extract_batch_id", "raw_load_timestamp"]

common_cols = [*base_cols, "standard_time"]

def to_quarantine(df, reason_code):
    return (df.select(*common_cols)
              .withColumn("_reject_reason", lit(reason_code))
              .withColumn("_quarantined_at", current_timestamp())
              .withColumn("_batch", lit(BATCH))
              .withColumn("_run_id", lit(RUN_ID)))

# ---------------------------------------------------------------------------
# STEP 1: Parse and normalize
# ---------------------------------------------------------------------------
base = bronze_time_df.select(*base_cols).withColumn(
    "clean_time", trim(col("source_time_string"))
).withColumn(
    "norm_time",
    regexp_replace(
        regexp_replace(col("clean_time"), r"(?i)a\.m\.", "AM"),
        r"(?i)p\.m\.", "PM"
    )
)

# ---------------------------------------------------------------------------
# STEP 2: Quarantine UNKNOWN precision rows
# ---------------------------------------------------------------------------
quarantine_parts.append(to_quarantine(
    base.filter(col("time_precision") == "UNKNOWN")
        .withColumn("standard_time", lit(None).cast("string")),
    "UNKNOWN_PRECISION"
))
base = base.filter(col("time_precision") != "UNKNOWN")

# ---------------------------------------------------------------------------
# STEP 3: Expand MINUTE times -> HH:mm:00 format for parsing
# For MINUTE precision: if numeric 4-digit, append '00'; if 12-hour, add ':00'
# ---------------------------------------------------------------------------
base = base.withColumn(
    "expanded_time",
    when(col("time_precision") == "MINUTE",
        when(col("norm_time").rlike(r"^\d{4}$"),
            concat(col("norm_time"), lit("00"))
        ).otherwise(
            regexp_replace(col("norm_time"), r"(?i)\s+(AM|PM)$", ":00 $1")
        )
    ).otherwise(col("norm_time"))
)

# ---------------------------------------------------------------------------
# STEP 4: Convert numeric -> HH:mm:ss / 12h AM/PM -> 24h
# ---------------------------------------------------------------------------
base = base.withColumn(
    "formatted_numeric",
    regexp_replace(lpad(col("expanded_time"), 6, "0"), r"(\d{2})(\d{2})(\d{2})", "$1:$2:$3")
).withColumn(
    "parsed_ampm",
    date_format(expr("try_to_timestamp(expanded_time, 'h:mm:ss a')"), "HH:mm:ss")
)

# ---------------------------------------------------------------------------
# STEP 5: Derive standard_time
# ---------------------------------------------------------------------------
base = base.withColumn(
    "standard_time",
    coalesce(
        when(col("expanded_time").rlike(r"^\d{1,2}:\d{2}:\d{2}$"), col("expanded_time")),
        col("parsed_ampm"),
        when(col("expanded_time").rlike(r"^\d{6}$"), col("formatted_numeric")),
    )
)

# Pad single-digit hour to 2 digits (e.g., '1:30:00' -> '01:30:00')
base = base.withColumn(
    "standard_time",
    when(
        col("standard_time").rlike(r"^\d:\d{2}:\d{2}$"),
        concat(lit("0"), col("standard_time"))
    ).otherwise(col("standard_time"))
)

# ---------------------------------------------------------------------------
# STEP 6: Quarantine NULL_TIME rows (parse failures)
# ---------------------------------------------------------------------------
quarantine_parts.append(to_quarantine(
    base.filter(col("standard_time").isNull()),
    "NULL_TIME"
))
base_valid = base.filter(col("standard_time").isNotNull())

# ---------------------------------------------------------------------------
# STEP 6b: Quarantine DUPLICATE_BATCH rows
# ---------------------------------------------------------------------------
quarantine_parts.append(to_quarantine(
    base_valid.filter(col("extract_batch_id").contains("DUP")),
    "DUPLICATE_BATCH"
))
base_valid = base_valid.filter(~col("extract_batch_id").contains("DUP"))

# ---------------------------------------------------------------------------
# STEP 7: Split SECOND / MINUTE, expand MINUTE -> 60 rows each
# ---------------------------------------------------------------------------
second_df = base_valid.filter(col("time_precision") == "SECOND").select(*common_cols)

minute_df = base_valid.filter(col("time_precision") == "MINUTE").select(*common_cols
).withColumn("sec", explode(sequence(lit(0), lit(59)))
).withColumn(
    "standard_time",
    concat_ws(":", substring(col("standard_time"), 1, 5), lpad(col("sec").cast("string"), 2, "0"))
).drop("sec")

# ---------------------------------------------------------------------------
# STEP 8: Union and deduplicate by standard_time
# ---------------------------------------------------------------------------
time_silver_base = second_df.union(minute_df).dropDuplicates(["standard_time"])

# ---------------------------------------------------------------------------
# STEP 9: Build final silver time dimension with derived fields
# ---------------------------------------------------------------------------
silver_time_df = time_silver_base.select(
    col("standard_time").alias("TimeValue"),
    expr("CAST(substring(standard_time, 1, 2) AS INT)").alias("HourID"),
    concat(substring(col("standard_time"), 1, 2), lit(":00")).alias("HourDesc"),
    expr("CAST(substring(standard_time, 4, 2) AS INT)").alias("MinuteID"),
    substring(col("standard_time"), 1, 5).alias("MinuteDesc"),
    expr("CAST(substring(standard_time, 7, 2) AS INT)").alias("SecondID"),
    col("standard_time").alias("SecondDesc"),
    when(
        (expr("CAST(substring(standard_time, 1, 2) AS INT)") >= 9) &
        (expr("CAST(substring(standard_time, 1, 2) AS INT)") < 16),
        lit(True)
    ).otherwise(lit(False)).cast(BooleanType()).alias("MarketHoursFlag"),
    when(
        (expr("CAST(substring(standard_time, 1, 2) AS INT)") >= 8) &
        (expr("CAST(substring(standard_time, 1, 2) AS INT)") < 17),
        lit(True)
    ).otherwise(lit(False)).cast(BooleanType()).alias("OfficeHoursFlag"),
    lit(BATCH).alias("_batch"),
    current_timestamp().alias("_load_ts"),
    lit(RUN_ID).alias("_run_id")
)

silver_time_count = silver_time_df.count()
print(f"Silver time record count: {silver_time_count}")

# ---------------------------------------------------------------------------
# Combine all quarantine parts
# ---------------------------------------------------------------------------
from functools import reduce
time_quarantine_all = reduce(lambda a, b: a.unionByName(b), quarantine_parts)
quarantine_time_count = time_quarantine_all.count()
print(f"Time records quarantined: {quarantine_time_count}")

In [0]:
# ---------------------------------------------------------------------------
# Write Silver Tables to Delta
# ---------------------------------------------------------------------------

# Write silver.date
silver_date_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.date")

log_pipeline_message(spark, RUN_ID, "INFO", "silver_date_and_time", f"silver.date written successfully with {silver_date_count} rows")

# Write silver.time (generated 86,400 seconds)
silver_time_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.time")

log_pipeline_message(spark, RUN_ID, "INFO", "silver_date_and_time", f"silver.time written successfully with {silver_time_count} rows")

# ---------------------------------------------------------------------------
# Write Quarantine Tables
# ---------------------------------------------------------------------------
date_quarantine_all.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.date")

time_quarantine_all.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.time")

log_pipeline_message(spark, RUN_ID, "INFO", "silver_date_and_time", f"Quarantine tables written: date={quarantine_date_count}, time={quarantine_time_count}")

print(f"Silver and quarantine tables written successfully.")
print(f"silver.date: {silver_date_count} rows | silver.time: {silver_time_count} rows")
print(f"quarantine.date: {quarantine_date_count} rows | quarantine.time: {quarantine_time_count} rows")

In [0]:
# ---------------------------------------------------------------------------
# Reconciliation and Audit Logging
# ---------------------------------------------------------------------------

# Build reconciliation DataFrame for reporting
from pyspark.sql import Row

recon_data = [
    Row(
        run_id=RUN_ID, batch_id=BATCH, domain=DOMAIN,
        table_name="date", source_layer="bronze", target_layer="silver",
        source_count=bronze_date_count, target_count=silver_date_count,
        quarantine_count=quarantine_date_count,
        variance=bronze_date_count - silver_date_count - quarantine_date_count,
        status="MATCH" if (bronze_date_count == silver_date_count + quarantine_date_count) else "MISMATCH"
    ),
    Row(
        run_id=RUN_ID, batch_id=BATCH, domain=DOMAIN,
        table_name="time", source_layer="bronze", target_layer="silver",
        source_count=bronze_time_count, target_count=silver_time_count,
        quarantine_count=quarantine_time_count,
        variance=bronze_time_count - silver_time_count - quarantine_time_count,
        status="MATCH" if (bronze_time_count == silver_time_count + quarantine_time_count) else "MISMATCH"
    )
]

recon_df = spark.createDataFrame(recon_data)
display(recon_df)

# ---------------------------------------------------------------------------
# Log reconciliation to operations.pipeline_recon_results using audit utility
# ---------------------------------------------------------------------------
log_pipeline_recon(
    spark, RUN_ID, BATCH, DOMAIN,
    table_name="date",
    source_layer="bronze",
    target_layer="silver",
    source_count=bronze_date_count,
    target_count=silver_date_count
)

log_pipeline_recon(
    spark, RUN_ID, BATCH, DOMAIN,
    table_name="time",
    source_layer="bronze",
    target_layer="silver",
    source_count=bronze_time_count,
    target_count=silver_time_count
)

# Log audit events
log_audit_event(spark, RUN_ID, BATCH, "silver", "date", "OVERWRITE", silver_date_count)
log_audit_event(spark, RUN_ID, BATCH, "silver", "time", "OVERWRITE", silver_time_count)
log_audit_event(spark, RUN_ID, BATCH, "quarantine", "date", "OVERWRITE", quarantine_date_count)
log_audit_event(spark, RUN_ID, BATCH, "quarantine", "time", "OVERWRITE", quarantine_time_count)

# End pipeline run
end_pipeline_run(spark, RUN_ID, "SUCCESS")
log_pipeline_message(spark, RUN_ID, "INFO", "silver_date_and_time", "Silver layer processing completed successfully for Date and Time")

print(f"Pipeline run {RUN_ID} completed. Reconciliation logged to operations.")